## Ансамбли и полносвязные нейронные сети
В этом ноутбуке вам нужно обучить модели на датасете классификации из предыдущего ноутбука и сравнить результаты. Вам будет предоставлен baseline, на основе которого вы будете доделывать предсказывающие модели. Оценка лабы будет зависеть от ROC-AUC на тестовых данных по следующим критериям:
\
AUC - на тестовых данных
- $AUC \leq 0.76$ - 0 баллов
- $0.76 < AUC \leq 0.77$ - 2 балла
- $0.77 < AUC \leq 0.78$ - 4 балла
- $0.78 < AUC \leq 0.79$ - 6 баллов
- $0.79 < AUC \leq 0.80$ - 8 баллов
- $AUC > 0.80$ - 10 баллов


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('german.csv', sep=';')
print(data.head())

X = data.iloc[:, 1:].to_numpy()
y = data.iloc[:, 0].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#    Creditability  Account_Balance  Duration_of_Credit_monthly  \
# 0              1                1                          18   
# 1              1                1                           9   
# 2              1                2                          12   
# 3              1                1                          12   
# 4              1                1                          12   

#    Payment_Status_of_Previous_Credit  Purpose  Credit_Amount  \
# 0                                  4        2           1049   
# 1                                  4        0           2799   
# 2                                  2        9            841   
# 3                                  4        0           2122   
# 4                                  4        0           2171   

#    Value_Savings_Stocks  Length_of_current_employment  Instalment_per_cent  \
# 0                     1                             2                    4   
# 1                     1                             3                    2   
# 2                     2                             4                    2   
# 3                     1                             3                    3   
# 4                     1                             3                    4   

#    Sex_Marital_Status  ...  Duration_in_Current_address  \
# 0                   2  ...                            4   
# 1                   3  ...                            2   
# 2                   2  ...                            4   
# 3                   3  ...                            2   
# 4                   3  ...                            4   

#    Most_valuable_available_asset  Age_years  Concurrent_Credits  \
# 0                              2         21                   3   
# 1                              1         36                   3   
# 2                              1         23                   3   
# 3                              1         39                   3   
# 4                              2         38                   1   

#    Type_of_apartment  No_of_Credits_at_this_Bank  Occupation  \
# 0                  1                           1           3   
# 1                  1                           2           3   
# 2                  1                           1           2   
# 3                  1                           2           2   
# 4                  2                           2           2   

#    No_of_dependents  Telephone  Foreign_Worker  
# 0                 1          1               1  
# 1                 2          1               1  
# 2                 1          1               1  
# 3                 2          1               2  
# 4                 1          1               2  

# [5 rows x 21 columns]

In [ ]:
plt.hist(y_train, bins=2, edgecolor='k')
plt.xticks([0, 1])
plt.xlabel('Class (0: Non-Creditworthy, 1: Creditworthy)')
plt.ylabel('Count')
plt.title('Distribution of Classes in Training Data')
plt.show()

In [ ]:
imort pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
data = pd.read_csv('german.csv', sep=';')

# Разделение на признаки и целевую переменную
X = data.drop('Creditability', axis=1)
y = data['Creditability']

# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# ОПТИМИЗИРОВАННЫЙ RANDOM FOREST


rf_param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [15, 20, 25, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', None]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_pred_proba = rf_best.predict_proba(X_test)[:, 1]
rf_roc_auc = roc_auc_score(y_test, rf_pred_proba)

print(f"Random Forest ROC AUC: {rf_roc_auc:.4f}")


# ОПТИМИЗИРОВАННЫЙ GRADIENT BOOSTING


gb_param_grid = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 0.9, 1.0],
    'max_features': ['sqrt', 'log2']
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

gb_grid.fit(X_train, y_train)
gb_best = gb_grid.best_estimator_
gb_pred_proba = gb_best.predict_proba(X_test)[:, 1]
gb_roc_auc = roc_auc_score(y_test, gb_pred_proba)

print(f"Gradient Boosting ROC AUC: {gb_roc_auc:.4f}")


# ОПТИМИЗИРОВАННАЯ НЕЙРОННАЯ СЕТЬ


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp_param_grid = {
    'hidden_layer_sizes': [(100,), (100, 50), (100, 50, 25)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01],
    'max_iter': [1000],
    'early_stopping': [True]
}

mlp_grid = GridSearchCV(
    MLPClassifier(random_state=42),
    mlp_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

mlp_grid.fit(X_train_scaled, y_train)
mlp_best = mlp_grid.best_estimator_
mlp_pred_proba = mlp_best.predict_proba(X_test_scaled)[:, 1]
mlp_roc_auc = roc_auc_score(y_test, mlp_pred_proba)

print(f"Neural Network ROC AUC: {mlp_roc_auc:.4f}")


# АНСАМБЛЕВАЯ МОДЕЛЬ (VOTING CLASSIFIER)


voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('gb', gb_best),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42))
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)
voting_pred_proba = voting_clf.predict_proba(X_test)[:, 1]
voting_roc_auc = roc_auc_score(y_test, voting_pred_proba)

print(f"Ensemble Voting ROC AUC: {voting_roc_auc:.4f}")


# УЛУЧШЕННЫЙ RANDOM FOREST


enhanced_rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

enhanced_rf.fit(X_train, y_train)
enhanced_rf_pred_proba = enhanced_rf.predict_proba(X_test)[:, 1]
enhanced_rf_roc_auc = roc_auc_score(y_test, enhanced_rf_pred_proba)

print(f"Enhanced Random Forest ROC AUC: {enhanced_rf_roc_auc:.4f}")



all_auc_scores = {
    'Random Forest': rf_roc_auc,
    'Gradient Boosting': gb_roc_auc,
    'Neural Network': mlp_roc_auc,
    'Ensemble Voting': voting_roc_auc,
    'Enhanced Random Forest': enhanced_rf_roc_auc
}

best_model = max(all_auc_scores, key=all_auc_scores.get)
best_auc = all_auc_scores[best_model]

print(f"\nBest model: {best_model} with ROC AUC: {best_auc:.4f}")

# Random Forest ROC AUC: 0.8286
# Gradient Boosting ROC AUC: 0.8323
# Neural Network ROC AUC: 0.8183
# Ensemble Voting ROC AUC: 0.8304
# Enhanced Random Forest ROC AUC: 0.8275

# Best model: Gradient Boosting with ROC AUC: 0.8323


In [ ]:
# Обучение оптимизированной MLP (Multi-Layer Perceptron) нейронной сети
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    alpha=0.001,
    learning_rate_init=0.01,
    max_iter=1000,
    early_stopping=True,
    random_state=42
)
mlp_model.fit(X_train_scaled, y_train)

# Прогноз вероятностей на тестовых данных
mlp_pred_proba = mlp_model.predict_proba(X_test_scaled)[:, 1]

# Расчет ROC AUC для MLP нейронной сети
mlp_roc_auc = roc_auc_score(y_test, mlp_pred_proba)

print(f"MLP (Neural Network) ROC AUC: {mlp_roc_auc:.4f}")

# MLP (Neural Network) ROC AUC: 0.8054

## Экспериментируйте
Для получения лучшего качества придется поэкспериментировать. Подсказка: попробуйте оптимизировать гиперпараметры модели